# [Introduction to Data Science](http://datascience-intro.github.io/1MS041-2026/)    
## 1MS041, 2026 
&copy;2026 Raazesh Sainudiin, Benny Avelin. [Attribution 4.0 International     (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)

# ProbSS 7 — Cleaning, joining, and testing without leakage

## What you will do

You will combine two local data sources, check the join, and learn every data
transformation from training rows only. You will compare random and spatial
test sets, evaluate one fixed regression method, and use a plot to decide what
data should be collected next.

The flight table is a teaching copy without embedded provenance or reuse
terms. Do not republish it as an authoritative source until those details have
been checked.


In [ ]:
from io import StringIO

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from pathlib import Path

def course_data(filename):
    candidates = (
        Path("data") / filename,
        Path("master/jp/data") / filename,
    )
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from master/jp "
        "or from the repository root."
    )

# Evaluation scale chosen before testing; it is not estimated
# from the held-out outcomes. Prices use the source table's units.
LOSS_SCALE = 2_000.0


## 1. Start with two local data sources

Source A is data/flights.csv. One row is a flight leg. `userCode` is an identifier; because source documentation is absent, we cannot verify whether or how it was pseudonymised. Treat it as potentially identifying and do not use it as a predictive feature.

Source B is the small course-maintained city-to-region registry embedded below. It exists to teach a many-to-one join and spatial holdout. It is not a substitute for an authoritative geographic registry.


In [ ]:
flights = pd.read_csv(course_data("flights.csv"))

city_registry_csv = StringIO(
    """city,macro_region
Recife (PE),Northeast
Florianopolis (SC),South
Brasilia (DF),Central-West
Aracaju (SE),Northeast
Salvador (BH),Northeast
Campo Grande (MS),Central-West
Sao Paulo (SP),Southeast
Natal (RN),Northeast
Rio de Janeiro (RJ),Southeast
"""
)
city_registry = pd.read_csv(city_registry_csv)

print("Flight rows:", len(flights))
print("Registry rows:", len(city_registry))


## 2. Clean and join the data before splitting


In [ ]:
flights.columns = flights.columns.str.strip()
for column in ["from", "to", "flightType", "agency"]:
    flights[column] = flights[column].astype(str).str.strip()
flights["date"] = pd.to_datetime(flights["date"], format="%m/%d/%Y", errors="coerce")
for column in ["price", "time", "distance"]:
    flights[column] = pd.to_numeric(flights[column], errors="coerce")

quality_before = {
    "rows": len(flights),
    "duplicate rows": int(flights.duplicated().sum()),
    "missing dates": int(flights["date"].isna().sum()),
    "nonpositive prices": int((flights["price"] <= 0).sum()),
    "nonpositive distances": int((flights["distance"] <= 0).sum()),
}

flights = flights.drop_duplicates()
flights = flights.dropna(
    subset=["date", "price", "time", "distance", "from", "to", "flightType", "agency"]
)
flights = flights[(flights["price"] > 0) & (flights["distance"] > 0) & (flights["time"] > 0)]

joined = flights.merge(
    city_registry.rename(
        columns={"city": "from", "macro_region": "origin_region"}
    ),
    on="from",
    how="left",
    validate="many_to_one",
    indicator="origin_join",
)
joined = joined.merge(
    city_registry.rename(
        columns={"city": "to", "macro_region": "destination_region"}
    ),
    on="to",
    how="left",
    validate="many_to_one",
    indicator="destination_join",
)

unmatched = (
    (joined["origin_join"] != "both")
    | (joined["destination_join"] != "both")
)
assert not unmatched.any(), "Resolve unmatched cities before modelling."
joined = joined.drop(columns=["origin_join", "destination_join"])

print("Quality before cleaning:", quality_before)
print("Rows after cleaning and joins:", len(joined))


The join validation prevents a duplicated registry key from silently multiplying flight rows. Checking unmatched keys prevents missing geography from being mistaken for an ordinary category.


## 3. Compare random and spatial test sets


In [ ]:
working = joined.sample(
    n=min(60_000, len(joined)),
    random_state=2026,
).sort_values("date").reset_index(drop=True)

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=2026,
)
group_train_idx, group_test_idx = next(
    group_splitter.split(working, groups=working["userCode"])
)
group_overlap = set(working.loc[group_train_idx, "userCode"]) & set(
    working.loc[group_test_idx, "userCode"]
)

time_cut = working["date"].quantile(0.80)
temporal_train = working["date"] < time_cut
temporal_test = ~temporal_train

spatial_test_region = "Southeast"
spatial_test = working["origin_region"].eq(spatial_test_region)
spatial_train = ~spatial_test

split_audit = pd.DataFrame(
    {
        "design": ["grouped by user", "temporal", "spatial by origin region"],
        "training rows": [
            len(group_train_idx),
            int(temporal_train.sum()),
            int(spatial_train.sum()),
        ],
        "test rows": [
            len(group_test_idx),
            int(temporal_test.sum()),
            int(spatial_test.sum()),
        ],
        "separation check": [
            f"user overlap = {len(group_overlap)}",
            (
                f"train ends {working.loc[temporal_train, 'date'].max().date()}, "
                f"test starts {working.loc[temporal_test, 'date'].min().date()}"
            ),
            f"test origin region = {spatial_test_region}",
        ],
    }
)
split_audit


Choose the split from the intended deployment. We use the spatial split below because the stated question is: how well does a model trained on other origin regions transfer to flights originating in the Southeast? A future-price question would instead require the temporal split. Repeated-user leakage would motivate the grouped split.

Decide how evaluation will work before inspecting held-out outcomes. We will report MAE, RMSE, and $R^2$. For a bounded-loss illustration, we also choose in advance

$$
L(y,\widehat y)=\min\left\{\frac{|y-\widehat y|}{2000},1\right\}.
$$

The value 2000 is a course reporting scale in the price units of this table; it is not estimated from the test set and is not claimed to be a universal business threshold. A real deployment must replace it by a domain-justified value fixed before final evaluation.


## 4. Learn every data transformation from the training set


In [ ]:
features = [
    "distance",
    "time",
    "flightType",
    "agency",
    "destination_region",
]
numeric_features = ["distance", "time"]
categorical_features = ["flightType", "agency", "destination_region"]

X_train = working.loc[spatial_train, features]
y_train = working.loc[spatial_train, "price"]
X_test = working.loc[spatial_test, features]
y_test = working.loc[spatial_test, "price"]

preprocess = ColumnTransformer(
    [
        ("numeric", StandardScaler(), numeric_features),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)
procedure = make_pipeline(preprocess, Ridge(alpha=10.0))
procedure.fit(X_train, y_train)
prediction = procedure.predict(X_test)

metrics = {
    "MAE": mean_absolute_error(y_test, prediction),
    "RMSE": mean_squared_error(y_test, prediction) ** 0.5,
    "R2": r2_score(y_test, prediction),
}
metrics


The transformations were fitted only on training rows. The test rows were not used to choose features, scaling, categories, the ridge penalty, the evaluation metrics, the clipped-loss definition, or its scale.


## 5. Choose the bounded loss before testing


In [ ]:
bounded_loss = np.minimum(
    np.abs(y_test.to_numpy() - prediction) / LOSS_SCALE,
    1.0,
)
alpha = 0.05
radius = np.sqrt(np.log(2 / alpha) / (2 * len(bounded_loss)))
nominal_interval = (
    max(0.0, bounded_loss.mean() - radius),
    min(1.0, bounded_loss.mean() + radius),
)

print(f"Clipping scale chosen before testing: {LOSS_SCALE:.0f} price units")
print(f"Mean clipped absolute loss: {bounded_loss.mean():.3f}")
print("Nominal Hoeffding interval if test losses were independent:", nominal_interval)
print(
    "Caution: repeated users/routes and the spatial sampling design mean that "
    "independence is not established, so this is an assumption check rather "
    "than a certified guarantee."
)


## 6. Let the plot guide the next step


In [ ]:
plot_frame = working.loc[spatial_test, ["distance", "flightType"]].copy()
plot_frame["residual"] = y_test.to_numpy() - prediction

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
take = np.linspace(0, len(plot_frame) - 1, min(1_500, len(plot_frame))).astype(int)
axes[0].scatter(
    plot_frame.iloc[take]["distance"],
    plot_frame.iloc[take]["residual"],
    s=10,
    alpha=0.4,
)
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set(
    xlabel="distance",
    ylabel="observed minus predicted price",
    title="Residuals in the spatial holdout",
)

groups = [
    values["residual"].to_numpy()
    for _, values in plot_frame.groupby("flightType")
]
labels = [name for name, _ in plot_frame.groupby("flightType")]
axes[1].boxplot(groups, tick_labels=labels, showfliers=False)
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set(
    xlabel="flight type",
    ylabel="residual",
    title="Systematic error by flight type",
)
plt.tight_layout()
plt.show()

print(plot_frame.groupby("flightType")["residual"].agg(["median", "mean", "count"]))


Decision rule: do not deploy the model for an origin region if a practically important residual trend remains by distance or flight type. Diagnose and revise on development data, then obtain a new untouched evaluation set; do not repeatedly tune to this test plot.

Data-responsibility check:

- provenance/licence: unresolved for the flight course copy; resolve before redistribution;
- privacy: `userCode` is an identifier that may be pseudonymous, but that status is unverified; treat it as potentially identifying and exclude it from features;
- minimisation: retain only fields needed for the stated prediction and split;
- integrity: validate key uniqueness and unmatched joins;
- dependence: users, routes, dates, and regions create groups;
- communication: report source units, the split population, metrics, and failed assumptions.


## Recap

Before you finish, make sure you can:

1. Explain what the quality log and join checks found.
2. Choose grouped, temporal, or spatial testing for a real use case and explain your choice.
3. Report the metrics and explain whether the bound's independence assumption is reasonable.
4. Use the plot to make a decision, then state what new data would be needed for a clean reevaluation.
